Defining a pipeline to convert yolo formatted labels into a coco label formatted json file.

In [15]:
from pathlib import Path

In [16]:

w = Path('C:\\Users\\Administrator\\Downloads\\Computer-Vision-Pipeline\\data\\weld_data')
for subdir in w.iterdir():
    if subdir.is_dir():
        print(subdir)

C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\annotations
C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\test
C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\train
C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\val
C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\val_new


In [17]:
W = str(w)

In [18]:
import os
import json
from PIL import Image

train_images_dir = W+"\\train\\images"
train_labels_dir = W+"\\train\\labels"

val_images_dir = W+"\\val\\images"
val_labels_dir = W+"\\val\\labels"

#categories defined for the weld_dataset, these should be consistent with the class indices in the label files, so alter based on the different dataset used.
coco = {
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 0, "name": "Pore"},
        {"id": 1, "name": "Inclusion"},
        {"id": 2, "name": "Undercut"},
        {"id": 3, "name": "Burn-through"},
        {"id": 4, "name": "Crack"},
        {"id": 5, "name": "Overlap"},
        {"id": 6, "name": "Reference Sample 1"},
        {"id": 7, "name": "Reference Sample 2"},
        {"id": 8, "name": "Reference Sample 3"},
        {"id": 9, "name": "Hidden Pore"},
        {"id": 10, "name": "Shrinkage Depression"},
        {"id": 11, "name": "Lack of Fusion"},
        {"id": 12, "name": "Incomplete Root Penetration"}
    ]
}




In [44]:
def convert_to_coco(train_images_dir, train_labels_dir, output_json, categories):
    coco_data = {
        "images": [],
        "annotations": [],
        "categories": categories
    }
    
    image_id = 1
    annotation_id = 1

    for image_file in os.listdir(train_images_dir):

        if not image_file.endswith((".jpg", ".png", ".jpeg")):
            continue

        image_path = os.path.join(train_images_dir, image_file)

        width, height = Image.open(image_path).size

        coco_data["images"].append({
            "id": image_id,
            "file_name": image_file,
            "width": width,
            "height": height
        })

        label_file = os.path.join(
            train_labels_dir,
            os.path.splitext(image_file)[0] + ".txt"
        )

        if os.path.exists(label_file):

            with open(label_file) as f:

                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.replace(",", " ").split()
                    if len(parts) != 5:
                        raise ValueError(
                            f"Unexpected label format in {label_file}: {line}"
                        )
                    cls, xc, yc, w, h = map(float, parts)

                    x_min = (xc - w / 2) * width
                    y_min = (yc - h / 2) * height
                    bbox_w = w * width
                    bbox_h = h * height

                    coco_data["annotations"].append({
                        "id": annotation_id,
                        "image_id": image_id,
                        "category_id": int(cls),
                        "bbox": [
                            x_min,
                            y_min,
                            bbox_w,
                            bbox_h
                        ],
                        "area": bbox_w * bbox_h,
                        "iscrowd": 0
                    })

                    annotation_id += 1

        image_id += 1

    with open(output_json, "w") as f:
        json.dump(coco_data, f)

In [ ]:
convert_to_coco(train_images_dir,train_labels_dir,"C:\\Users\\Administrator\\Downloads\\Computer-Vision-Pipeline\\data\\weld_data\\annotations\\instances_train.json", coco["categories"])

convert_to_coco(val_images_dir,val_labels_dir,"C:\\Users\\Administrator\\Downloads\\Computer-Vision-Pipeline\\data\\weld_data\\annotations\\instances_val.json", coco["categories"])

Converting License Plate Labels into COCO Annotations

In [21]:
L = str(Path('C:\\Users\\Administrator\\Downloads\\Computer-Vision-Pipeline\\data\\license_plate_detection'))

In [45]:
train_images_dir = L+"\\train\\images"
train_labels_dir = L+"\\train\\labels"

val_images_dir = L+"\\valid\\images"
val_labels_dir = L+"\\valid\\labels"

test_images_dir = L+"\\test\\images"
test_labels_dir = L+"\\test\\labels"

#categories defined for the weld_dataset, these should be consistent with the class indices in the label files, so alter based on the different dataset used.
coco = {
    "images": [],
    "annotations": [],
    "categories": [
        {"id": 1, "name": "License_Plate"},
    ]
}

In [43]:
convert_to_coco(train_images_dir,train_labels_dir,"C:\\Users\\Administrator\\Downloads\\Computer-Vision-Pipeline\\data\\license_plate_detection\\annotations\\instances_train.json", coco["categories"])
convert_to_coco(val_images_dir,val_labels_dir,"C:\\Users\\Administrator\\Downloads\\Computer-Vision-Pipeline\\data\\license_plate_detection\\annotations\\instances_val.json", coco["categories"])
convert_to_coco(test_images_dir,test_labels_dir,"C:\\Users\\Administrator\\Downloads\\Computer-Vision-Pipeline\\data\\license_plate_detection\\annotations\\instances_test.json", coco["categories"])